In [15]:
# Mutual Fund Data Analysis Script
# This script fetches mutual fund data, calculates performance metrics, and exports to CSV

from mftool import Mftool
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

# Initialize Mftool
mf = Mftool()

# 1. Get all scheme codes
print("Fetching all mutual fund scheme codes...")
all_scheme_codes = mf.get_scheme_codes()
master_funds = pd.DataFrame(all_scheme_codes.items())
master_funds.columns = ['scheme_code', 'scheme_name']
master_funds = master_funds[1:].reset_index(drop=True)
print(f"Total schemes found: {len(master_funds)}")

# 2. Function to calculate CAGR
def calculate_cagr(start_nav, end_nav, years):
    if years <= 0 or start_nav <= 0:
        return np.nan
    return ((end_nav / start_nav) ** (1 / years) - 1) * 100

# 3. Function to get current NAV
def get_current_nav(scheme_code):
    try:
        nav = mf.get_scheme_quote(scheme_code)['nav']
        return float(nav)
    except:
        return np.nan

# 4. Fetch scheme details for all funds
print("\nFetching detailed scheme information (this may take a while)...")
scheme_details_list = []

for idx, code in enumerate(master_funds['scheme_code']):
    try:
        details = mf.get_scheme_details(code)
        scheme_details_list.append(details)

        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(master_funds)} schemes...")
            time.sleep(1)  # Rate limiting

    except Exception as e:
        scheme_details_list.append(None)
        continue

# 5. Create detailed dataframe
print("\nProcessing scheme details...")
detailed_data = []

for details in scheme_details_list:
    if details:
        detailed_data.append({
            'fund_house': details.get('fund_house', ''),
            'scheme_type': details.get('scheme_type', ''),
            'scheme_category': details.get('scheme_category', ''),
            'scheme_code': details.get('scheme_code', ''),
            'scheme_name': details.get('scheme_name', ''),
            'scheme_start_date': details.get('scheme_start_date', {}),
            'start_date': details.get('scheme_start_date', {}).get('date', ''),
            'nav': details.get('scheme_start_date', {}).get('nav', np.nan)
        })

final_df = pd.DataFrame(detailed_data)
print(f"Schemes with details: {len(final_df)}")

# 6. Convert start_date to datetime
final_df['start_date'] = pd.to_datetime(final_df['start_date'], format='%d-%m-%Y', errors='coerce')
final_df['nav'] = pd.to_numeric(final_df['nav'], errors='coerce')

# 7. Calculate CAGR for 3 and 5 years
print("\nCalculating 3-year and 5-year CAGR...")
today = datetime.now()

def calc_cagr_years(row, years):
    if pd.isna(row['start_date']):
        return np.nan
    fund_age = (today - row['start_date']).days / 365.25
    if fund_age < years:
        return np.nan

    try:
        current_nav = get_current_nav(row['scheme_code'])
        if pd.isna(current_nav):
            return np.nan
        return calculate_cagr(row['nav'], current_nav, years)
    except:
        return np.nan

final_df['cagr_3yrs'] = final_df.apply(lambda x: calc_cagr_years(x, 3), axis=1)
final_df['cagr_5yrs'] = final_df.apply(lambda x: calc_cagr_years(x, 5), axis=1)

# 8. Update current NAV
print("\nUpdating current NAV values...")
final_df['current_nav'] = final_df['scheme_code'].apply(get_current_nav)

# 9. Calculate additional metrics
print("\nCalculating additional performance metrics...")

# Sharpe Ratio (simplified estimation)
def calculate_sharpe(cagr):
    if pd.isna(cagr):
        return np.nan
    risk_free_rate = 6.5  # Assumed risk-free rate
    volatility = abs(cagr) * 0.15  # Simplified volatility estimate
    return (cagr - risk_free_rate) / volatility if volatility > 0 else np.nan

final_df['sharpe_ratio'] = final_df['cagr_3yrs'].apply(calculate_sharpe)

# Drawdown (simplified)
final_df['drawdown'] = final_df.apply(
    lambda x: ((x['current_nav'] - x['nav']) / x['nav'] * 100) if x['nav'] > 0 else np.nan, 
    axis=1
)

# Volatility (simplified estimate)
final_df['volatility'] = final_df['cagr_3yrs'].apply(lambda x: abs(x) * 0.15 if not pd.isna(x) else np.nan)

# Market capture ratios (simplified estimates)
final_df['upmarket_capture'] = final_df['cagr_3yrs'].apply(
    lambda x: max(0, x) * 0.9 if not pd.isna(x) else np.nan
)
final_df['downmarket_capture'] = final_df['cagr_3yrs'].apply(
    lambda x: min(0, x) * 0.15 if not pd.isna(x) else np.nan
)

# 10. Extract plan type and scheme type from scheme name
def extract_plan_info(scheme_name):
    if 'Direct' in scheme_name:
        plan_type = 'Direct'
    else:
        plan_type = 'Regular'

    if 'Growth' in scheme_name:
        scheme_type = 'Growth'
    elif 'IDCW' in scheme_name or 'Dividend' in scheme_name:
        scheme_type = 'IDCW'
    else:
        scheme_type = 'Other'

    return plan_type, scheme_type

final_df[['plan_type', 'scheme_type']] = final_df['scheme_name'].apply(
    lambda x: pd.Series(extract_plan_info(x))
)

# 11. Calculate fund age
final_df['fund_age_years'] = final_df['start_date'].apply(
    lambda x: (today - x).days / 365.25 if not pd.isna(x) else np.nan
)

# 12. Clean and reorder columns
columns_order = [
    'fund_house', 'scheme_code', 'scheme_name', 'scheme_category',
    'plan_type', 'scheme_type', 'start_date', 'fund_age_years',
    'nav', 'current_nav', 'cagr_3yrs', 'cagr_5yrs',
    'sharpe_ratio', 'volatility', 'drawdown',
    'upmarket_capture', 'downmarket_capture'
]

final_output = final_df[columns_order].copy()

# 13. Export to CSV
output_filename = f'mutual_fund_analysis_{datetime.now().strftime("%Y%m%d")}.csv'
final_output.to_csv(output_filename, index=False)
print(f"\nData exported to: {output_filename}")

# 14. Display summary statistics
print("\n=== Summary Statistics ===")
print(f"Total schemes analyzed: {len(final_output)}")
print(f"Schemes with 3-year CAGR: {final_output['cagr_3yrs'].notna().sum()}")
print(f"Schemes with 5-year CAGR: {final_output['cagr_5yrs'].notna().sum()}")
print(f"\nAverage 3-year CAGR: {final_output['cagr_3yrs'].mean():.2f}%")
print(f"Average 5-year CAGR: {final_output['cagr_5yrs'].mean():.2f}%")

print("\n=== Top 10 Schemes by 3-Year CAGR ===")
print(final_output.nlargest(10, 'cagr_3yrs')[['scheme_name', 'cagr_3yrs', 'cagr_5yrs']])

print("\nAnalysis complete!")

Fetching all mutual fund scheme codes...
Total schemes found: 4895

Fetching detailed scheme information (this may take a while)...
Processed 100/4895 schemes...
Processed 200/4895 schemes...


KeyboardInterrupt: 